# 계류장 자동 표시 + YOLO26x 학습 (Colab GPU)

사람이 박스를 그리지 않고, **큰 AI 모델이 박스를 그리게 한 뒤** 그 박스로 YOLO26x를 학습합니다.
화면(AERO INSIGHT)에서는 학습된 YOLO만 돌아갑니다.

1. **사진 모으기**: 위키미디어 공용(Wikimedia Commons)에서 공항 항공 사진을 받습니다. 재사용 가능한 라이선스(CC0·CC BY·CC BY-SA·퍼블릭 도메인)만 받고, 출처·작가·라이선스는 `attribution.csv`에 남깁니다.
2. **박스 모델 비교**: Grounding DINO(base)와 OWLv2(large)를 테스트 사진(뮌헨·인천·두바이)과 받은 사진에 돌려 그림으로 비교합니다.
3. **자동 표시**: 고른 모델(`TEACHER`)로 모든 사진에 박스를 그립니다. 작은 물체를 찾도록 사진을 조각내서 찾습니다.
4. **학습**: 자동 표시 사진 + (선택) 사람이 표시한 Roboflow CCTV 사진으로 YOLO26x를 학습합니다.

**찾는 종류 4가지**: 비행기(aircraft), 탑승교(jet_bridge), 지상 차량(ground_vehicle), 사람(person).
탑승교를 따로 배우게 해서 게이트를 비행기로 잘못 보는 문제를 줄이려는 목적입니다. 차량 세부 종류(급유차·토잉카 등)는 자동 표시가 부정확해서 하나로 묶습니다.

**주의**: 학습된 YOLO는 박스를 그려 준 모델보다 더 정확해지기 어렵습니다. 검증 성적도 '자동 박스를 얼마나 따라 하나'일 뿐 정답 성적이 아닙니다. 그래서 마지막에 테스트 사진 3장에서 기존 YOLO26x와 그림으로 비교합니다.

**실행**: 런타임 유형 A100 → `런타임 → 모두 실행`. 처음에 Roboflow 키를 묻습니다(CCTV 데이터를 섞지 않으려면 그냥 Enter).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics==8.4.215 roboflow "transformers>=4.51" timm
from getpass import getpass
RF_KEY = getpass("Roboflow Private API Key (CCTV 데이터를 안 섞으려면 Enter): ").strip()

## 1. 위키미디어 공용에서 공항 항공 사진 받기

In [ ]:
import csv, re, time, hashlib
from pathlib import Path
import requests

RAW = Path("/content/commons"); RAW.mkdir(parents=True, exist_ok=True)
S = requests.Session()
S.headers["User-Agent"] = "apron-dataset-builder/0.1 (student research project)"
API = "https://commons.wikimedia.org/w/api.php"
QUERIES = ["airport apron aerial", "aerial view airport terminal aircraft", "airport aerial photograph",
           "aircraft parked apron aerial view", "Flughafen Vorfeld Luftbild", "aéroport vue aérienne avions",
           "airport from above planes gates", "aerial photo international airport terminal"]
OK_LICENSE = re.compile(r"^(cc0|cc[- ]by(-sa)?[- ]?[0-9.]*|public domain|pd.*)$", re.I)
MAX_IMAGES = 600

def search(q, offset):
    p = dict(action="query", format="json", generator="search", gsrsearch=f"{q} filetype:bitmap", gsrnamespace=6,
             gsrlimit=50, gsroffset=offset, prop="imageinfo", iiprop="url|extmetadata|size|mime", iiurlwidth=2048)
    return S.get(API, params=p, timeout=30).json()

meta, seen = [], set()
for q in QUERIES:
    for off in range(0, 300, 50):
        try:
            pages = search(q, off).get("query", {}).get("pages", {})
        except Exception as e:
            print("검색 실패", q, off, e); break
        if not pages:
            break
        for pg in pages.values():
            ii = (pg.get("imageinfo") or [{}])[0]
            em = ii.get("extmetadata", {})
            lic = em.get("LicenseShortName", {}).get("value", "").strip()
            if pg["title"] in seen or ii.get("mime") not in ("image/jpeg", "image/png") or ii.get("width", 0) < 1200:
                continue
            if not OK_LICENSE.match(lic) or "nc" in lic.lower() or "nd" in lic.lower():
                continue
            seen.add(pg["title"])
            artist = re.sub("<[^>]+>", "", em.get("Artist", {}).get("value", ""))[:200]
            meta.append(dict(title=pg["title"], url=ii.get("thumburl") or ii["url"], page=ii.get("descriptionurl", ""),
                             license=lic, artist=artist))
        time.sleep(0.5)
print("후보", len(meta), "장")

saved = []
for m in meta[:MAX_IMAGES]:
    name = hashlib.md5(m["title"].encode()).hexdigest()[:12] + ".jpg"
    try:
        r = S.get(m["url"], timeout=60)
        if r.ok and len(r.content) > 50_000:
            (RAW / name).write_bytes(r.content); m["file"] = name; saved.append(m)
    except Exception:
        pass
    time.sleep(0.2)
with open("/content/attribution.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["file", "title", "license", "artist", "page", "url"]); w.writeheader(); w.writerows(saved)
print("받은 사진", len(saved), "장")

## 2. 테스트 사진(뮌헨·인천·두바이) 준비

In [ ]:
import base64
from pathlib import Path
TEST = Path('/content/test'); TEST.mkdir(exist_ok=True)
TEST_B64 = {
    'munich': '...',
    'test1': '...',
    'dubai': '...',
}
for n, s in TEST_B64.items():
    (TEST / f'{n}.jpg').write_bytes(base64.b64decode(s))
print(sorted(p.name for p in TEST.iterdir()))

## 3. 박스 그리는 모델 (Grounding DINO, OWLv2)

사진을 1024px 조각(20% 겹침) + 전체 사진으로 나눠 찾고, 같은 종류끼리 겹친 박스를 합칩니다.

In [ ]:
import torch, numpy as np
from PIL import Image
from torchvision.ops import batched_nms
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Owlv2Processor, Owlv2ForObjectDetection

DEV = "cuda"
CLASSES = ["aircraft", "jet_bridge", "ground_vehicle", "person"]

def to_class(label):
    l = str(label).lower()
    if "bridge" in l: return 1
    if "plane" in l or "aircraft" in l: return 0
    if "person" in l or "worker" in l: return 3
    if "vehicle" in l or "truck" in l or "car" in l or "cart" in l or "tug" in l: return 2
    return None

gd_proc = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-base")
gd = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-base").to(DEV).eval()
GD_TEXT = "airplane. jet bridge. vehicle. person."

ow_proc = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
ow = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble").to(DEV).eval()
OW_QUERIES = ["a photo of an airplane", "a jet bridge at an airport gate", "an airport ground vehicle", "a person"]

@torch.no_grad()
def gd_detect(img, thr=0.3):
    inp = gd_proc(images=img, text=GD_TEXT, return_tensors="pt").to(DEV)
    out = gd(**inp)
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25,
                                                           target_sizes=[img.size[::-1]])[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25,
                                                           target_sizes=[img.size[::-1]])[0]
    labels = r.get("text_labels", r["labels"])
    rows = [(*b.tolist(), float(s), to_class(l)) for b, s, l in zip(r["boxes"], r["scores"], labels)]
    return [x for x in rows if x[5] is not None]

@torch.no_grad()
def ow_detect(img, thr=0.2):
    inp = ow_proc(text=[OW_QUERIES], images=img, return_tensors="pt").to(DEV)
    out = ow(**inp)
    s = max(img.size)   # OWLv2 는 사진을 정사각형으로 채워 넣는다
    post = getattr(ow_proc, "post_process_grounded_object_detection", None) or ow_proc.post_process_object_detection
    r = post(out, threshold=thr, target_sizes=torch.tensor([[s, s]], device=DEV))[0]
    return [(*b.tolist(), float(sc), int(l)) for b, sc, l in zip(r["boxes"], r["scores"], r["labels"])]

def tiled(detect, img, tile=1024, overlap=0.2):
    W, H = img.size
    rows = list(detect(img))
    step = int(tile * (1 - overlap))
    if max(W, H) > tile * 1.25:
        xs = list(range(0, max(W - tile, 0) + 1, step)); ys = list(range(0, max(H - tile, 0) + 1, step))
        if xs[-1] + tile < W: xs.append(W - tile)
        if ys[-1] + tile < H: ys.append(H - tile)
        for y in ys:
            for x in xs:
                for x1, y1, x2, y2, sc, c in detect(img.crop((x, y, min(x + tile, W), min(y + tile, H)))):
                    rows.append((x1 + x, y1 + y, x2 + x, y2 + y, sc, c))
    if not rows:
        return np.zeros((0, 6))
    t = torch.tensor(rows, dtype=torch.float32)
    t = t[((t[:, 2] - t[:, 0]) * (t[:, 3] - t[:, 1])) < 0.4 * W * H]   # 사진 대부분을 덮는 박스는 버림
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), 0.5)
    return t[keep].numpy()

## 4. 비교 그림: 기존 YOLO26x(COCO) / Grounding DINO / OWLv2

In [ ]:
import random, cv2, matplotlib.pyplot as plt
from ultralytics import YOLO

coco = YOLO("yolo26x.pt")
COLORS = [(255, 60, 60), (60, 160, 255), (255, 200, 0), (0, 220, 120)]

def draw(img, boxes):
    a = np.array(img).copy()
    for x1, y1, x2, y2, sc, c in boxes:
        cv2.rectangle(a, (int(x1), int(y1)), (int(x2), int(y2)), COLORS[int(c)], 2)
    return a

def coco_boxes(img):
    r = coco.predict(img, imgsz=1280, conf=0.2, classes=[4, 0, 2, 5, 7], verbose=False)[0]
    m = {4: 0, 0: 3, 2: 2, 5: 2, 7: 2}
    return [(*b[:4], b[4], m[int(b[5])]) for b in r.boxes.data.tolist()]

random.seed(0)
samples = sorted(TEST.glob("*.jpg")) + random.sample(sorted(RAW.glob("*.jpg")), min(5, len(list(RAW.glob("*.jpg")))))
for p in samples:
    img = Image.open(p).convert("RGB")
    res = {"YOLO26x(COCO)": coco_boxes(img), "Grounding DINO": tiled(gd_detect, img), "OWLv2": tiled(ow_detect, img)}
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    for ax, (k, b) in zip(axes, res.items()):
        cnt = np.bincount([int(x[5]) for x in b], minlength=4)
        ax.imshow(draw(img, b)); ax.axis("off")
        ax.set_title(f"{k}  aircraft {cnt[0]} bridge {cnt[1]} vehicle {cnt[2]} person {cnt[3]}", fontsize=13)
    plt.suptitle(p.name + "   (red aircraft, blue jet bridge, yellow vehicle, green person)"); plt.tight_layout(); plt.show()

## 5. 자동 표시

`TEACHER`로 고른 모델로 받은 사진 전체에 박스를 그립니다. 비행기가 2대 이상이거나 탑승교가 있는 사진만 남깁니다(공항이 아닌 사진 거르기).

In [ ]:
TEACHER = "gdino"   # 비교 그림을 보고 "owlv2" 로 바꿀 수 있다
detect = gd_detect if TEACHER == "gdino" else ow_detect
OUT = Path("/content/auto");
import shutil, random
shutil.rmtree(OUT, ignore_errors=True)
files = sorted(RAW.glob("*.jpg")); random.seed(0); random.shuffle(files)
kept = []
for i, p in enumerate(files):
    try:
        img = Image.open(p).convert("RGB")
    except Exception:
        continue
    b = tiled(detect, img)
    cnt = np.bincount(b[:, 5].astype(int), minlength=4) if len(b) else np.zeros(4, int)
    if cnt[0] < 2 and cnt[1] < 1:
        continue
    kept.append((p, img.size, b))
    if i % 50 == 0: print(i, "/", len(files), "남긴 사진", len(kept), flush=True)
n_val = max(int(len(kept) * 0.15), 1)
for j, (p, (W, H), b) in enumerate(kept):
    split = "valid" if j < n_val else "train"
    (OUT / split / "images").mkdir(parents=True, exist_ok=True); (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy(p, OUT / split / "images" / p.name)
    lines = [f"{int(c)} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}" for x1, y1, x2, y2, s, c in b]
    (OUT / split / "labels" / (p.stem + ".txt")).write_text("\n".join(lines))
tot = np.bincount(np.concatenate([k[2][:, 5].astype(int) for k in kept]), minlength=4) if kept else np.zeros(4)
print("자동 표시 사진", len(kept), "장 (검증", n_val, ") 박스:", dict(zip(CLASSES, tot.tolist())))

## 6. (선택) 사람이 표시한 Roboflow CCTV 사진 섞기

In [ ]:
import yaml
if RF_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=RF_KEY)
    MAP = {"plane": 0, "aircraft": 0, "jet bridge": 1, "boarding bridge": 1, "passenger": 3, "staff": 3, "person": 3}
    VEH = {"car", "truck", "bus", "passenger bus", "fire truck", "water service", "baggage truck", "container transport",
           "belt loader", "fuel truck", "pushback truck", "aircraft tractor", "boarding stairway", "aircraft ladder",
           "catering truck", "lift truck", "container loader", "gpu"}
    for ws, name in [("datalabeling-0nprc", "handling"), ("airport-azgu2", "airport_90")]:
        proj = rf.workspace(ws).project(name)
        ver = max(int(str(v.version).split("/")[-1]) for v in proj.versions())
        root = Path(proj.version(ver).download("yolov8", location=f"/content/raw_rf/{name}").location)
        names = yaml.safe_load((root / "data.yaml").read_text())["names"]
        names = list(names.values()) if isinstance(names, dict) else names
        remap = {i: (MAP.get(n.lower()) if n.lower() not in VEH else 2) for i, n in enumerate(names)}
        for split in ("train", "valid"):
            for img in (root / split / "images").glob("*"):
                lab = root / split / "labels" / (img.stem + ".txt")
                rows = []
                for line in (lab.read_text().splitlines() if lab.exists() else []):
                    q = line.split()
                    if q and remap.get(int(q[0])) is not None:
                        rows.append(" ".join([str(remap[int(q[0])])] + q[1:]))
                (OUT / split / "images").mkdir(parents=True, exist_ok=True); (OUT / split / "labels").mkdir(parents=True, exist_ok=True)
                shutil.copy(img, OUT / split / "images" / f"rf_{name}_{img.name}")
                (OUT / split / "labels" / f"rf_{name}_{img.stem}.txt").write_text("\n".join(rows))
        print(name, "추가")
(OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
                                               "names": dict(enumerate(CLASSES))}))
for s in ("train", "valid"):
    print(s, len(list((OUT / s / "images").glob("*"))), "장")

## 7. YOLO26x 학습 (이미지 1280, 작은 차량·사람용)

In [ ]:
model = YOLO("yolo26x.pt")
model.train(data="/content/auto/data.yaml", epochs=150, patience=30, imgsz=1280, batch=-1, cos_lr=True,
            close_mosaic=15, degrees=5, scale=0.5, fliplr=0.5, project="/content/runs/auto", name="yolo26x",
            exist_ok=True, seed=0, deterministic=True, plots=True)

## 8. 테스트 사진 비교: 기존 YOLO26x vs 새 모델

In [ ]:
new = YOLO("/content/runs/auto/yolo26x/weights/best.pt")
for p in sorted(TEST.glob("*.jpg")):
    img = Image.open(p).convert("RGB")
    r = new.predict(img, imgsz=1280, conf=0.25, verbose=False)[0]
    nb = [(*b[:4], b[4], int(b[5])) for b in r.boxes.data.tolist()]
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    for ax, (k, b) in zip(axes, {"YOLO26x (COCO)": coco_boxes(img), "NEW (auto-labeled)": nb}.items()):
        cnt = np.bincount([int(x[5]) for x in b], minlength=4)
        ax.imshow(draw(img, b)); ax.axis("off"); ax.set_title(f"{k}  aircraft {cnt[0]} bridge {cnt[1]} vehicle {cnt[2]} person {cnt[3]}", fontsize=13)
    plt.suptitle(p.name); plt.tight_layout(); plt.show()

## 9. 가중치·기록 내려받기 (안 받아지면 왼쪽 📁 파일에서 직접 다운로드)

In [ ]:
import shutil
from google.colab import files
shutil.copy("/content/runs/auto/yolo26x/weights/best.pt", "/content/apron_auto_yolo26x.pt")
shutil.copy("/content/attribution.csv", "/content/runs/auto/yolo26x/attribution.csv")
shutil.make_archive("/content/apron_auto_log", "zip", "/content/runs/auto/yolo26x")
files.download("/content/apron_auto_yolo26x.pt"); files.download("/content/apron_auto_log.zip")